In [5]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
def load_documents(DATA_PATH):
    doc_loader = PyPDFDirectoryLoader(DATA_PATH)
    return doc_loader.load()

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def split_document(documents: list[Document]):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=80,
        length_function=len,
        is_separator_regex=False,
    )
    return text_splitter.split_documents(documents)

In [10]:
chunks = split_document(document)
print(chunks[150])

In [ ]:
from langchain_community.embeddings.ollama import OllamaEmbeddings
def get_embedding_function():
    embeddings = OllamaEmbeddings(model="mistral:latest")
    return embeddings

In [ ]:

def calculate_chunk_ids(chunks):

    last_page_id = None
    current_chunk_index = 0

    for chunk in chunks:
        source = chunk.metadata.get("source")
        page = chunk.metadata.get("page")
        current_page_id = f"{source}:{page}"
        if current_page_id == last_page_id:
            current_chunk_index += 1
        else:
            current_chunk_index = 0
        chunk_id = f"{current_page_id}:{current_chunk_index}"
        last_page_id = current_page_id
        chunk.metadata["id"] = chunk_id

    return chunks


In [15]:
from langchain_community.vectorstores import Chroma
def add_to_chroma(chunks: list[Document],CHROMA_PATH):
# Load the existing database.
    db = Chroma(
        persist_directory=CHROMA_PATH, embedding_function=get_embedding_function()
    )

    # Calculate Page IDs.
    chunks_with_ids = calculate_chunk_ids(chunks)

    # Add or Update the documents.
    existing_items = db.get(include=[])  # IDs are always included by default
    existing_ids = set(existing_items["ids"])
    print(f"Number of existing documents in DB: {len(existing_ids)}")

    # Only add documents that don't exist in the DB.
    new_chunks = []
    for chunk in chunks_with_ids:
        if chunk.metadata["id"] not in existing_ids:
            new_chunks.append(chunk)

    if len(new_chunks):
        print(f" Adding new documents: {len(new_chunks)}")
        new_chunk_ids = [chunk.metadata["id"] for chunk in new_chunks]
        db.add_documents(new_chunks, ids=new_chunk_ids)
        db.persist()
    else:
        print(" No new documents to add")


In [16]:
def main():

    documents = load_documents(r'Data')
    chunks = split_document(documents)
    add_to_chroma(chunks,r'chroma1.2')

In [17]:


if __name__ == "__main__":
    main()

Number of existing documents in DB: 0
 Adding new documents: 1944


C:\Users\NOAMAN\AppData\Local\Temp\ipykernel_19764\1868107427.py:26: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


In [ ]:
PROMPT_TEMPLATE = """
Answer the question based only on the following context:

{context}

---

Answer the question based on the above context: {question}
"""


In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms.ollama import Ollama

def query_rag(query_text: str,CHROMA_PATH):
    # Prepare the DB.
    embedding_function = get_embedding_function()
    db = Chroma(persist_directory=CHROMA_PATH, embedding_function=embedding_function)

    # Search the DB.
    results = db.similarity_search_with_score(query_text, k=5)

    context_text = "\n\n---\n\n".join([doc.page_content for doc, _score in results])
    prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
    prompt = prompt_template.format(context=context_text, question=query_text)
    # print(prompt)

    model = Ollama(model="mistral")
    response_text = model.invoke(prompt)

    sources = [doc.metadata.get("id", None) for doc, _score in results]
    formatted_response = f"Response: {response_text}\nSources: {sources}"
    print(formatted_response)
    return response_text

In [ ]:
response = query_rag(
    query_text="What is Azure?",
    CHROMA_PATH="./chroma1.2"  # Your ChromaDB path
)
print(response)

In [41]:
# Data model for grading
class GradeDocuments(BaseModel):
    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )

print(" Initializing Ollama grader")
llm = Ollama(model="mistral", temperature=0)

system_prompt = """You are a grader assessing relevance of a retrieved document to a user question. 
If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant.
It does not need to be a stringent test. The goal is to filter out erroneous retrievals.

You MUST respond with ONLY a JSON object in this exact format:
{
    "binary_score": "yes" or "no"
}

DO NOT include any other text or explanations."""

grade_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Retrieved document: \n\n{document}\n\nUser question: {question}"),
])

print(" Ollama grader ready!")

 Initializing Ollama grader
 Ollama grader ready!


In [42]:
def grade_document_ollama(document_text, question):

    try:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Retrieved document: \n\n{document_text}\n\nUser question: {question}"}
        ]
        
        # Get response from Ollama
        response = llm.invoke(json.dumps(messages))
        
        # Parse response (try to extract JSON)
        response_text = response.strip()
        
        # Look for JSON in the response
        if "{" in response_text and "}" in response_text:
            json_start = response_text.find("{")
            json_end = response_text.rfind("}") + 1
            json_str = response_text[json_start:json_end]
            
            try:
                result = json.loads(json_str)
                binary_score = result.get("binary_score", "no").lower()
                is_relevant = binary_score == "yes"
                
                return {
                    "grade": binary_score,
                    "is_relevant": is_relevant,
                    "document_preview": document_text[:150] + "..." if len(document_text) > 150 else document_text
                }
            except json.JSONDecodeError:
                # Fallback: check if response contains "yes" or "no"
                if "yes" in response_text.lower():
                    return {"grade": "yes", "is_relevant": True, "document_preview": document_text[:150] + "..."}
                else:
                    return {"grade": "no", "is_relevant": False, "document_preview": document_text[:150] + "..."}
        else:
            # Simple keyword matching as fallback
            if "yes" in response_text.lower():
                return {"grade": "yes", "is_relevant": True, "document_preview": document_text[:150] + "..."}
            else:
                return {"grade": "no", "is_relevant": False, "document_preview": document_text[:150] + "..."}
                
    except Exception as e:
        print(f" Error: {e}")
        return {"grade": "error", "is_relevant": False, "document_preview": document_text[:150] + "..."}

In [43]:
def grade_documents_for_rag(documents_list, question, show_progress=True):
    relevant_docs = []
    all_results = []
    
    print(f"Grading {len(documents_list)} documents...")
    print(f"Question: '{question}'\n")
    
    for i, doc in enumerate(documents_list, 1):
        if show_progress:
            print(f"  [{i}/{len(documents_list)}] Processing...", end="\r")
        
        result = grade_document_ollama(doc, question)
        result["full_document"] = doc
        
        all_results.append(result)
        
        if result["is_relevant"]:
            relevant_docs.append(doc)
    
    if show_progress:
        print(" " * 50, end="\r")  # Clear line
    
    # Print summary
    print(f"\n Grading Complete!")
    print(f" Relevant documents: {len(relevant_docs)}/{len(documents_list)}")
    
    # Show which documents were relevant
    print("\n Relevance Breakdown:")
    for i, result in enumerate(all_results, 1):
        status = " Relevant" if result["is_relevant"] else " Not Relevant"
        print(f"{i}. {status} - {result['document_preview']}")
    
    return {
        "relevant_documents": relevant_docs,
        "all_results": all_results,
        "question": question,
        "total_documents": len(documents_list),
        "relevant_count": len(relevant_docs)
    }

In [45]:
import json
def get_rag_documents(query, chroma_path, k=10):
    # Your existing code here
    embedding_function = get_embedding_function()  
    db = Chroma(persist_directory=chroma_path, embedding_function=embedding_function)
    results = db.similarity_search_with_score(query, k=k)
    return results

# grader to filter documents
CHROMA_PATH = "./chroma1.2"  # Your ChromaDB path
user_question = "What is Azure?"

print(" Retrieving documents from ChromaDB")
results = get_rag_documents(user_question, CHROMA_PATH, k=10)
documents = [doc.page_content for doc, _score in results]

print(f" Retrieved {len(documents)} documents\n")

print(" Step 2: Grading document relevance...")
grading_results = grade_documents_for_rag(documents, user_question)

print("\n Step 3: Using filtered documents for RAG...")
if grading_results["relevant_documents"]:
    # Create context from relevant documents (like your original code)
    context_text = "\n\n---\n\n".join(grading_results["relevant_documents"][:5])
    
    # Now you can use this context_text with your existing prompt
    print(f"Filtered context ready ({len(context_text)} characters)")
    print(f"\nFirst 500 chars of context:\n{context_text[:500]}...")
else:
    print(" No relevant documents found for this question!")

 Retrieving documents from ChromaDB
 Retrieved 10 documents

 Step 2: Grading document relevance...
Grading 10 documents...
Question: 'What is Azure?'

                                                  
 Grading Complete!
 Relevant documents: 3/10

 Relevance Breakdown:
1.  Not Relevant - Collect & search across data sources from multiple systems, centralised in
a single data store to easily identify the root cause of operational issues...
2.  Relevant - monitoring. It is also a key part of Cortana Analytics Suite & works with Azure SQL Data Warehouse, Power BI &
Data Factory. This gives you a complete...
3.  Not Relevant - Balancer to support scale-out & high-availability for both Internet-facing &
internal-only web front ends.
4.  Relevant - appropriate for companies or
individuals using Microsoft Azure
in non-production environment
or for trial and evaluation
LEARN MORE → LEARN MORE → LEA...
5.  Not Relevant - queries against a large database – & when you want to invoke a service
wh

In [48]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_community.llms import Ollama

class GradeDocuments(BaseModel):
    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )

parser = PydanticOutputParser(pydantic_object=GradeDocuments)

system = """You are a grader assessing relevance of a retrieved document to a user question.
If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant.
Give ONLY the final answer in the required format.
{format_instructions}
"""

grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved document:\n\n{document}\n\nUser question: {question}")
    ]
).partial(format_instructions=parser.get_format_instructions())

llm = Ollama(model="mistral", temperature=0)

retrieval_grader = grade_prompt | llm | parser


In [77]:
def doc_fil(docs):
    docs_to_use = []

    for doc, score in docs:
        res = retrieval_grader.invoke({
            "question": question,
            "document": doc.page_content
        })
        if res.binary_score == "yes":
            docs_to_use.append(doc)

    return docs_to_use


In [60]:
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import Ollama
from langchain_community.llms import Ollama

# Prompt
system = """You are an assistant for question-answering tasks.
Answer the question based upon your knowledge.
Use three-to-five sentences maximum and keep the answer concise."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved documents:\n\n<docs>{documents}</docs>\n\nUser question: <question>{question}</question>"),
    ]
)

# Output parser
output_parser = StrOutputParser()

# Build the chain
rag_chain = prompt | llm | output_parser


In [78]:
def generation(question, docs):
    return rag_chain.invoke({
        "documents": format_docs(docs),
        "question": question
    })


In [79]:
question = "what is Azure?"

raw_docs = db.similarity_search_with_score(question, k=5)
filtered_docs = doc_fil(raw_docs)

answer = generation(question, filtered_docs)
print(answer)


 Azure refers to a cloud computing service provided by Microsoft. It offers various services such as Infrastructure as a Service (IaaS), Platform as a Service (PaaS), and Software as a Service (SaaS). The main purpose of Azure is to help businesses reduce errors, provide unified management, and balance access and control through automation.


In [82]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_community.llms import Ollama


class GradeHallucinations(BaseModel):
    binary_score: str = Field(
        description="Answer is grounded in the facts, 'yes' or 'no'"
    )


parser = PydanticOutputParser(pydantic_object=GradeHallucinations)

system = """You are a grader assessing whether an LLM generation is grounded in
a set of retrieved facts.
Give a binary score 'yes' or 'no'.

{format_instructions}
"""

hallucination_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", 
         "Set of facts:\n\n<facts>{documents}</facts>\n\n"
         "LLM generation:\n<generation>{generation}</generation>")
    ]
).partial(format_instructions=parser.get_format_instructions())

llm = Ollama(model="mistral", temperature=0)

hallucination_grader = hallucination_prompt | llm | parser

response = hallucination_grader.invoke({
    "documents": format_docs(docs_to_use),
    "generation": answer
})

print(response)


binary_score='yes'
